## 1.1

Pressed=ONW
Released=Ready (ON)
Press again=OFFW
Released=Ready (OFF)

Pressed: btn=0
Not pressed: btn=1

In [ ]:
# from machine import Pin
import time

# Pins
btn = Pin(7, Pin.IN, Pin.PULL_UP)   # grounding button
led = Pin(20, Pin.OUT)

# States
S_OFF = 0
S_ON_WAIT_RELEASE = 1
S_ON = 2
S_OFF_WAIT_RELEASE = 3

state = S_OFF

# Clock: 20 Hz → period = 50 ms
PERIOD = 0.05

while True:
    b = btn.value()  # 1 = not pressed, 0 = pressed

    if state == S_OFF:
        led.value(0)
        if b == 0:  # pressed
            state = S_ON_WAIT_RELEASE

    elif state == S_ON_WAIT_RELEASE:
        led.value(1)
        if b == 1:  # released
            state = S_ON

    elif state == S_ON:
        led.value(1)
        if b == 0:  # pressed again
            state = S_OFF_WAIT_RELEASE

    elif state == S_OFF_WAIT_RELEASE:
        led.value(0)
        if b == 1:  # released
            state = S_OFF

    time.sleep(PERIOD)

## 1.2
### Inputs
- alarm
- button
### Outputs
- siren
- light
### States
- Idle
- Alarm_Activated (ALARM)
- Button_Acknowledged (ACK_ACTIVE)
- NoButton_DeActivated (ACK_WAIT_CLEAR)

In [ ]:
# from machine import Pin
import time

# Inputs (PULL-UP → active LOW)
btn = Pin(7, Pin.IN, Pin.PULL_UP)
alarm = Pin(9, Pin.IN, Pin.PULL_UP)

# Outputs
led = Pin(22, Pin.OUT)
siren = Pin(20, Pin.OUT)

# States
IDLE = 0
ALARM = 1
ACK_ACTIVE = 2
ACK_WAIT_CLEAR = 3

state = IDLE

# Clock
PERIOD = 0.05  # 20 Hz

# Blink helper
blink = 0
blink_counter = 0
BLINK_RATE = 10  # toggle every 10 cycles (~0.5s)

while True:
    b = btn.value()      # 0 = pressed
    a = alarm.value()    # 0 = active

    # ---- STATE MACHINE ----
    if state == IDLE:
        led.value(0)
        siren.value(0)

        if a == 0:  # alarm triggered
            state = ALARM

    elif state == ALARM:
        led.value(1)
        siren.value(1)

        if b == 0:  # acknowledge
            state = ACK_ACTIVE
        elif a == 1:  # alarm cleared before acknowledge
            state = ACK_WAIT_CLEAR

    elif state == ACK_ACTIVE:
        siren.value(0)

        # blinking LED
        blink_counter += 1
        if blink_counter >= BLINK_RATE:
            blink ^= 1
            blink_counter = 0
        led.value(blink)

        if a == 1:  # alarm cleared
            state = IDLE

    elif state == ACK_WAIT_CLEAR:
        led.value(1)
        siren.value(0)

        if b == 0:  # user acknowledges after alarm cleared
            state = IDLE

    time.sleep(PERIOD)